[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/channel_extraction/your_area_basin_profiles.ipynb)

# Basins, channels and long profiles for your own area

In this notebook you choose the place. You provide two small GeoPackage files:

1. a **polygon** around the area you want to study, and
2. one or more **points** at the outlets (the downstream ends) of the river basins you are interested in.

The notebook then:
1. Installs `lsdtt3` (the command-line analysis tools) and `lsdviztools3` (the plotting package).
2. Downloads a 30 m DEM covering your polygon with `lsdtt-fetch-raster`.
3. Makes a hillshade with `lsdtt-raster-preprocessing`.
4. Extracts the basin upstream of each outlet point with `lsdtt-basin-extraction`.
5. Extracts the channels in those basins, with the elevation and the distance along the channel for every channel pixel, with `lsdtt-chi-analysis`.
6. Makes a map of the basins and channels over the hillshade, and a plot of the channel **long profiles** (elevation against distance upstream).

If you just want to see it working first, set `INPUT_MODE = "example"` in the settings cell: this uses two example files (Glen Coe and Glen Etive in the Scottish Highlands) from the `example_data` folder next to this notebook.

Each `lsdtt3` program is controlled by a small **parameter file**: a text file of `key: value` lines. We write these files from Python and then run the programs.

## Setup

### First set up condacolab

This is a bit of an annoying step that takes around 2 minutes.

**Note:** `condacolab.install()` restarts the Colab runtime. You will see a message saying the session crashed; this is expected. Just carry on running the cells below.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

Now we install `pygmt`. GMT stands for Generic Mapping Tools, and `lsdviztools3` uses it to make maps. This is the slowest step (around a minute).

In [ ]:
!mamba install pygmt

### Get lsdviztools3

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdviztools3-0.1.0-py3-none-any.whl

We install it with its `charts` extra, which adds `matplotlib` for the long-profile plot. The conda setup above replaces Colab's usual Python packages, so `matplotlib` may otherwise be missing. (Keep the quotes: they stop the shell from misreading the square brackets.)

In [ ]:
!pip install "lsdviztools3-0.1.0-py3-none-any.whl[charts]"

### Get the lsdtt3 command-line tools

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdtt3-backend-linux-x86_64-core-v0.5.2.tar.gz

In [ ]:
!tar -xzf lsdtt3-backend-linux-x86_64-core-v0.5.2.tar.gz

Now we tell the system where to find the `lsdtt3` programs and the libraries they need. If your runtime restarts later, run this cell again.

In [ ]:
import os

root = "/content/lsdtt3-backend-linux-x86_64-core-v0.5.2"
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's older proj.db breaks rasterio and pyproj.

Check that it works: this prints the version of `lsdtt3`.

In [ ]:
!lsdtt-chi-analysis -v

## Making your two input files in QGIS

You need two GeoPackage (`.gpkg`) files. Any coordinate system is fine: the notebook converts everything for you.

* **Area polygon.** In QGIS: *Layer > Create Layer > New GeoPackage Layer...*, choose geometry type **Polygon**, then use *Toggle Editing* and *Add Polygon Feature* to draw around your area. Draw generously: every basin you want must fit **completely** inside it, from its outlet all the way up to its drainage divides.
* **Outlet points.** Make a second GeoPackage layer with geometry type **Point** and click one point at the downstream end of each basin you want, on or right next to the river. The points must be inside the polygon.

Save both edits (*Save Layer Edits*). If you put both layers in one GeoPackage file, the notebook reads only the first layer, so it is simplest to make two separate files.

If your area file has several polygons they are merged into one area. Every point in the outlet file becomes a basin outlet.

## Your settings

**This is the only cell you normally need to edit.** Run it after changing anything.

There are three ways to give the notebook your files (`INPUT_MODE`):

* `"upload"`: the next cell shows a **Choose Files** button, twice: first pick your **area polygon** GeoPackage, then your **outlet points** GeoPackage. The files go into `/content`, Colab's working folder. Uploaded files disappear when the Colab session ends.
* `"paths"`: the files are already on the Colab machine, and you type their paths into `AREA_FILE` and `OUTLETS_FILE`. For example, you can drag them into the Files panel on the left (they end up in `/content`), or keep them on Google Drive: run `from google.colab import drive; drive.mount("/content/drive")` in a new cell, and use paths like `"/content/drive/MyDrive/my_area.gpkg"`.
* `"example"`: download the two example files (`example_area.gpkg` and `example_outlets.gpkg`) from the `example_data` folder of the lsdtt3_notebooks repository. The outlets file is deliberately in Web Mercator (EPSG:3857), a different coordinate system from the polygon (EPSG:4326), to show that any coordinate system works.

About the other settings:
* `THRESHOLD_CONTRIBUTING_PIXELS`: a channel starts where this many pixels drain to a point. With 30 m pixels, 500 pixels is 500 x 900 m² = 0.45 km². Smaller values give more, shorter channels.
* `BUFFER_KM`: the DEM is downloaded for a box around your polygon, plus this margin.
* The area limits guard against asking a free Colab machine for too much. See the note under Step 1.

In [ ]:
# ============================ YOUR SETTINGS ============================

# How do you want to supply the two GeoPackages? "upload", "paths" or "example"
INPUT_MODE = "upload"

# Only used when INPUT_MODE = "paths"
AREA_FILE = "/content/my_area.gpkg"        # polygon(s) around your study area
OUTLETS_FILE = "/content/my_outlets.gpkg"  # point(s) at the basin outlets

# Prefix for every file the notebook writes (letters, numbers, _ ; no spaces)
RUN_NAME = "myarea"

# Margin (km) added around your polygon when downloading the DEM
BUFFER_KM = 1.0

# Channels start where this many pixels drain to a point
THRESHOLD_CONTRIBUTING_PIXELS = 500

# DEM grid spacing in metres (the Copernicus GLO-30 DEM is about 30 m)
GRID_SPACING = 30

# Size limits for the downloaded box, in km²
WARN_AREA_KM2 = 2500      # print a warning above this
MAX_AREA_KM2 = 10000      # stop above this ...
ALLOW_LARGE_AREA = False  # ... unless you set this to True

# ======================================================================

## Get the input files

This cell uploads, downloads or finds your two files, depending on `INPUT_MODE`.

In [ ]:
import os
import urllib.request

EXAMPLE_URL = ("https://raw.githubusercontent.com/LSDtopotools/lsdtt3_notebooks/"
               "main/channel_extraction/example_data/")

if INPUT_MODE == "upload":
    from google.colab import files

    print("Step 1 of 2: choose the GeoPackage with your AREA POLYGON")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f"Please choose exactly one file for the area (got {len(uploaded)}).")
    AREA_FILE = os.path.abspath(next(iter(uploaded)))

    print("Step 2 of 2: choose the GeoPackage with your OUTLET POINTS")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f"Please choose exactly one file for the outlets (got {len(uploaded)}).")
    OUTLETS_FILE = os.path.abspath(next(iter(uploaded)))

elif INPUT_MODE == "example":
    for name in ["example_area.gpkg", "example_outlets.gpkg"]:
        urllib.request.urlretrieve(EXAMPLE_URL + name, name)
    AREA_FILE = os.path.abspath("example_area.gpkg")
    OUTLETS_FILE = os.path.abspath("example_outlets.gpkg")
    if RUN_NAME == "myarea":
        RUN_NAME = "example"

elif INPUT_MODE != "paths":
    raise ValueError(f'INPUT_MODE must be "upload", "paths" or "example", not {INPUT_MODE!r}.')

for label, path in [("Area polygon", AREA_FILE), ("Outlet points", OUTLETS_FILE)]:
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"{label} file not found: {path}\n"
            "Check the path in the settings cell (the Files panel on the left shows "
            "what is on the Colab machine), or use INPUT_MODE = 'upload'.")
    print(f"{label} file: {path}")

if not RUN_NAME.replace("_", "").isalnum():
    raise ValueError("RUN_NAME may only contain letters, numbers and underscores.")

## Read and check the inputs

We read both files with `geopandas` and check them:

* The area file must contain polygons, and the outlet file points. Swapping the two files is an easy mistake, and the message says so.
* Both files must have a coordinate system (CRS). We convert them to longitude and latitude (EPSG:4326) to work out the box to download and to pass the outlets to `lsdtt3`.
* Every outlet must be inside the area polygon.
* The download box must not be too big for Colab.

We also choose the **UTM zone** for the analysis from the centre of your polygon. UTM is a projected coordinate system in metres. Flow routing and distances along channels need metres, not degrees. (`lsdtt-fetch-raster` would pick the same zone itself if `target_epsg` were left at 0, which uses the centre of the box; we choose it here so we know it and can use it for the maps.)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import make_valid
from shapely.geometry import box


def read_first_layer(path, what):
    """Read the first layer of a GeoPackage and check it has a CRS."""
    try:
        layers = gpd.list_layers(path)
    except Exception as err:
        raise ValueError(f"Could not open the {what} file {path}. Is it a GeoPackage? ({err})")
    layers = layers[layers["geometry_type"].notna()]
    if layers.empty:
        raise ValueError(f"The {what} file {path} has no layers with geometry.")
    layer = layers["name"].iloc[0]
    if len(layers) > 1:
        print(f"NOTE: the {what} file has {len(layers)} layers "
              f"({', '.join(layers['name'])}). Using the first one: '{layer}'.")
    gdf = gpd.read_file(path, layer=layer)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty]
    if gdf.empty:
        raise ValueError(f"The {what} layer '{layer}' has no features. Did you save your edits in QGIS?")
    if gdf.crs is None:
        raise ValueError(f"The {what} layer '{layer}' has no coordinate reference system (CRS). "
                         "In QGIS, set the layer CRS (Layer Properties > Source) and save it again.")
    return gdf, layer


def check_geometry(gdf, allowed, what, expected):
    found = sorted(set(gdf.geom_type))
    if not set(found) <= set(allowed):
        hint = ""
        if what == "area" and "Point" in found:
            hint = " It looks like the outlet points file. Did you swap the two files?"
        if what == "outlet" and ("Polygon" in found or "MultiPolygon" in found):
            hint = " It looks like the area polygon file. Did you swap the two files?"
        raise ValueError(f"The {what} file must contain {expected}, but it contains {', '.join(found)}.{hint}")


# --- Area polygon ------------------------------------------------------
area_gdf, area_layer = read_first_layer(AREA_FILE, "area")
check_geometry(area_gdf, {"Polygon", "MultiPolygon"}, "area", "polygons")
print(f"Area: {len(area_gdf)} polygon feature(s) in {area_gdf.crs.to_string()}")
area_ll = area_gdf.to_crs(4326)
aoi_ll = make_valid(area_ll.geometry.union_all())   # all polygons merged into one area

# --- UTM zone from the polygon centre ----------------------------------
lon_c, lat_c = aoi_ll.centroid.x, aoi_ll.centroid.y
if abs(lat_c) > 80:
    raise ValueError("This notebook uses UTM coordinates, which do not work near the poles (above 80°).")
utm_zone = int(np.floor((lon_c + 180.0) / 6.0)) % 60 + 1
EPSG = (32600 if lat_c >= 0 else 32700) + utm_zone
CRS_STR = f"EPSG:{EPSG}"
print(f"Polygon centre: longitude {lon_c:.4f}, latitude {lat_c:.4f}")
print(f"Using UTM zone {utm_zone}{'N' if lat_c >= 0 else 'S'}: {CRS_STR}")

aoi_utm = gpd.GeoSeries([aoi_ll], crs=4326).to_crs(EPSG)

# --- Download box: polygon + buffer, in longitude/latitude ---------------
west, south, east, north = aoi_utm.buffer(BUFFER_KM * 1000.0).to_crs(4326).total_bounds
box_km2 = gpd.GeoSeries([box(west, south, east, north)], crs=4326).to_crs(EPSG).area.iloc[0] / 1e6
n_pixels = box_km2 * 1e6 / GRID_SPACING**2
print(f"Download box: west {west:.4f}, east {east:.4f}, south {south:.4f}, north {north:.4f}")
print(f"Box area: {box_km2:.0f} km² (about {n_pixels / 1e6:.1f} million {GRID_SPACING} m pixels)")

if box_km2 > MAX_AREA_KM2 and not ALLOW_LARGE_AREA:
    raise ValueError(
        f"The download box is {box_km2:.0f} km², above MAX_AREA_KM2 = {MAX_AREA_KM2}. "
        "A free Colab machine has about 12 GB of memory and 2 processors: a box this big "
        "is slow to download, may run out of memory during flow routing, and makes maps "
        "with too much detail to see. Draw a smaller polygon around just your basins, or "
        "set ALLOW_LARGE_AREA = True if you really want to try.")
if box_km2 > WARN_AREA_KM2:
    print(f"WARNING: this is a large area (more than {WARN_AREA_KM2} km²). "
          "Each step will take longer, and the map will be crowded. "
          "Consider a smaller polygon around just the basins you need.")

# --- Outlet points ------------------------------------------------------
outlets_gdf, outlets_layer = read_first_layer(OUTLETS_FILE, "outlet")
check_geometry(outlets_gdf, {"Point", "MultiPoint"}, "outlet", "points")
print(f"Outlets: {len(outlets_gdf)} feature(s) in {outlets_gdf.crs.to_string()}")
outlets_gdf = outlets_gdf.explode(index_parts=False).reset_index(drop=True)

# Number the outlets 1, 2, 3 ... in the order they are stored in the file.
outlets_ll = outlets_gdf.to_crs(4326)
outlets = pd.DataFrame({
    "outlet_id": np.arange(1, len(outlets_ll) + 1),
    "latitude": outlets_ll.geometry.y,
    "longitude": outlets_ll.geometry.x,
})
outlets_utm = gpd.GeoDataFrame(outlets, geometry=outlets_ll.geometry, crs=4326).to_crs(EPSG)

outside = outlets_utm[~outlets_utm.within(aoi_utm.iloc[0])]
if len(outside) > 0:
    print(outside[["outlet_id", "longitude", "latitude"]].to_string(index=False))
    raise ValueError(
        f"{len(outside)} outlet point(s) (listed above) are outside the area polygon. "
        "Move them inside it, or draw a bigger polygon. Remember that the whole basin "
        "upstream of each outlet must also fit inside the polygon.")

OUTLETS_CSV = f"{RUN_NAME}_outlets.csv"
outlets.to_csv(OUTLETS_CSV, index=False, float_format="%.7f")
print(f"\nAll {len(outlets)} outlets are inside the area. Written to {OUTLETS_CSV}:")
print(outlets.to_string(index=False))


def check_outputs(*names):
    """Stop with a clear message if an lsdtt3 program did not write its outputs."""
    missing = [n for n in names if not os.path.isfile(n)]
    if missing:
        raise RuntimeError(f"Expected output file(s) not found: {missing}. "
                           "Scroll up and read the messages from the lsdtt3 program.")
    for n in names:
        print(f"OK: {n}")

## Step 1: Download a DEM

`lsdtt-fetch-raster` downloads a DEM for a box given in longitude and latitude. We use:

* `dem_source: cop30_aws`: the Copernicus GLO-30 DEM (about 30 m resolution), served from AWS. No account or API key is needed.
* `target_epsg`: the UTM zone chosen above, so the DEM is in metres.
* `grid_spacing: 30`: a 30 m output grid.

The output is called `<write_fname>_DEM.tif`.

**Why is there a size limit?** A free Colab machine has about 12 GB of memory and two processors. A 50 km x 50 km box at 30 m is about 2.8 million pixels, which is quick. Much bigger areas take longer to download and route, can run out of memory, and give maps with too much detail to read.

In [ ]:
with open(f"{RUN_NAME}_fetch.param", "w") as f:
    f.write(f"write_fname: {RUN_NAME}\n")
    f.write(f"west: {west:.6f}\n")
    f.write(f"east: {east:.6f}\n")
    f.write(f"south: {south:.6f}\n")
    f.write(f"north: {north:.6f}\n")
    f.write("dem_source: cop30_aws\n")
    f.write(f"target_epsg: {EPSG}\n")
    f.write(f"grid_spacing: {GRID_SPACING}\n")

!lsdtt-fetch-raster ./ {RUN_NAME}_fetch.param

check_outputs(f"{RUN_NAME}_DEM.tif")

## Step 2: Fill the DEM and make a hillshade

`lsdtt-raster-preprocessing` fills pits in the DEM (`write_fill_raster`), so that water can flow off the landscape everywhere, and computes a hillshade from the filled DEM (`write_hillshade_raster`) with the sun in the northwest (`hillshade_azimuth: 315`), 45 degrees above the horizon.

Outputs: `<RUN_NAME>_fill.tif` and `<RUN_NAME>_hillshade.tif`. The next two steps start from the filled DEM, so they can skip filling (`raster_is_filled: true`).

In [ ]:
with open(f"{RUN_NAME}_hillshade.param", "w") as f:
    f.write(f"read_fname: {RUN_NAME}_DEM.tif\n")
    f.write(f"write_fname: {RUN_NAME}\n")
    f.write("write_fill_raster: true\n")
    f.write("write_hillshade_raster: true\n")
    f.write("hillshade_azimuth: 315\n")
    f.write("hillshade_altitude: 45\n")

!lsdtt-raster-preprocessing ./ {RUN_NAME}_hillshade.param

check_outputs(f"{RUN_NAME}_fill.tif", f"{RUN_NAME}_hillshade.tif")

## Step 3: Extract the basins upstream of your outlets

`lsdtt-basin-extraction` routes flow over the filled DEM and extracts the basin draining to each outlet (`select_basins_from_outlets`). The outlets are read from the CSV file we wrote above, which has `latitude` and `longitude` columns (`basin_outlet_fname`).

**Snapping.** A point clicked in QGIS is rarely exactly on the DEM's river pixel. With `basin_outlet_snapping_method: downstream`, each point is moved downhill, following the flow, until it reaches a pixel with at least `basin_outlet_contributing_pixels_threshold` upstream pixels. We set that to the channel threshold, so each outlet lands on the channel network. A point on a hillslope next to a river is therefore moved into that river.

Outputs:
* `<RUN_NAME>_junction_basin_polygons.fgb`: one polygon per basin.
* `<RUN_NAME>_selected_basin_outlets.csv`: the snapped outlet of each basin with its area (`basin_area_km2`).

In [ ]:
with open(f"{RUN_NAME}_basins.param", "w") as f:
    f.write(f"read_fname: {RUN_NAME}_fill.tif\n")
    f.write(f"write_fname: {RUN_NAME}\n")
    f.write("raster_is_filled: true\n")
    f.write(f"threshold_contributing_pixels: {THRESHOLD_CONTRIBUTING_PIXELS}\n")
    f.write("select_basins_from_outlets: true\n")
    f.write(f"basin_outlet_fname: {OUTLETS_CSV}\n")
    f.write("basin_outlet_snapping_method: downstream\n")
    f.write(f"basin_outlet_contributing_pixels_threshold: {THRESHOLD_CONTRIBUTING_PIXELS}\n")
    f.write("write_basin_outlines: true\n")
    f.write("write_basin_outlines_as_polygon: true\n")
    f.write("write_selected_basin_outlets_csv: true\n")

!lsdtt-basin-extraction ./ {RUN_NAME}_basins.param

check_outputs(f"{RUN_NAME}_junction_basin_polygons.fgb", f"{RUN_NAME}_selected_basin_outlets.csv")

## Step 4: Extract the channels and their profiles

`lsdtt-chi-analysis` extracts the channel network upstream of each outlet, using the same outlets, snapping and channel threshold as Step 3. It is designed for *chi analysis* (a way of comparing channel steepness between rivers), but here we only use its **data maps**, which list every channel pixel with its elevation and its distance along the channels from the basin outlet. That is exactly what a long profile needs.

* `write_chi_data_maps: true` with `chi_data_maps_write_vector_format: csv` writes `<RUN_NAME>_chi_data_maps.csv`, one row per channel pixel with `latitude`, `longitude`, `elevation`, `flow_distance` (metres upstream from the outlet), `drainage_area`, `basin_key` and `source_key` (which channel, from its source, the pixel belongs to). It also writes the outlets (`<RUN_NAME>_chi_basin_outlets.csv`).
* `write_chi_data_map_lines: true` writes the channels as lines, one per source, in `<RUN_NAME>_chi_data_map_lines.fgb`.
* `n_largest_basins: 0` and `min_channel_pixels: 0` keep all your basins, however small.
* `m_over_n: 0.45` is only needed for the chi values, which we do not use here.

**Important:** this program only keeps **complete** basins: basins that do not touch the edge of the DEM. If your basin runs off the edge of the downloaded area, it has no profile. The next cell tells you if that happened.

In [ ]:
with open(f"{RUN_NAME}_channels.param", "w") as f:
    f.write(f"read_fname: {RUN_NAME}_fill.tif\n")
    f.write(f"write_fname: {RUN_NAME}\n")
    f.write("raster_is_filled: true\n")
    f.write(f"threshold_contributing_pixels: {THRESHOLD_CONTRIBUTING_PIXELS}\n")
    f.write(f"basin_outlet_fname: {OUTLETS_CSV}\n")
    f.write("basin_outlet_snapping_method: downstream\n")
    f.write(f"basin_outlet_contributing_pixels_threshold: {THRESHOLD_CONTRIBUTING_PIXELS}\n")
    f.write("n_largest_basins: 0\n")
    f.write("min_channel_pixels: 0\n")
    f.write("m_over_n: 0.45\n")
    f.write("write_chi_data_maps: true\n")
    f.write("chi_data_maps_write_vector_format: csv\n")
    f.write("write_chi_data_map_lines: true\n")

!lsdtt-chi-analysis ./ {RUN_NAME}_channels.param

## Match the results to your outlets

The two programs number their basins in their own ways. Here we give every basin the number of the outlet point it came from (1, 2, 3 ... in the order of the points in your file), and check which basins are complete.

In [ ]:
# Basins from Step 3. The polygons are not stored in the same order as the rows of
# the selected-outlets CSV, so we join them on the junction number of each basin.
basins = gpd.read_file(f"{RUN_NAME}_junction_basin_polygons.fgb").to_crs(EPSG)
sel = pd.read_csv(f"{RUN_NAME}_selected_basin_outlets.csv")
snapped = gpd.GeoDataFrame(sel, geometry=gpd.points_from_xy(sel.longitude, sel.latitude),
                           crs=4326).to_crs(EPSG)
poly_area = basins.area / 1e6
rows = []
for _, s in snapped.iterrows():
    candidates = basins.index[basins.junction_number == s.basin_junction_number]
    if len(candidates) == 0:
        raise RuntimeError(f"No basin polygon for junction {s.basin_junction_number}. "
                           "Scroll up to the messages from lsdtt-basin-extraction.")
    # Two outlets on the same stretch of river share a junction number; pick by area
    rows.append((poly_area[candidates] - s.basin_area_km2).abs().idxmin())
basins = basins.loc[rows].reset_index(drop=True)
basins["area_km2"] = sel["basin_area_km2"].values

# Each outlet point drains into its basin, so the point lies inside the basin
# polygon; of the points inside, the one closest to the snapped outlet is the one
# that made the basin.
basin_ids = []
for poly, outlet in zip(basins.geometry, snapped.geometry):
    near = outlets_utm[outlets_utm.within(poly.buffer(2 * GRID_SPACING))]
    if near.empty:
        near = outlets_utm
    basin_ids.append(int(near.loc[near.distance(outlet).idxmin(), "outlet_id"]))
basins["outlet_id"] = basin_ids
snapped["outlet_id"] = basin_ids
snapped["snap_distance_m"] = [
    snapped.geometry.iloc[i].distance(
        outlets_utm.loc[outlets_utm.outlet_id == oid, "geometry"].iloc[0])
    for i, oid in enumerate(basin_ids)]

# Channels from Step 4 (only complete basins). Match their outlets to the snapped outlets.
nodes = pd.DataFrame()
chan_lines = gpd.GeoDataFrame()
if os.path.isfile(f"{RUN_NAME}_chi_data_maps.csv"):
    nodes = pd.read_csv(f"{RUN_NAME}_chi_data_maps.csv")
    chi_out = pd.read_csv(f"{RUN_NAME}_chi_basin_outlets.csv")
    chi_out = gpd.GeoDataFrame(chi_out, geometry=gpd.points_from_xy(chi_out.longitude, chi_out.latitude),
                               crs=4326).to_crs(EPSG)
    key_to_id = {}
    for key, pt in zip(chi_out.basin_key, chi_out.geometry):
        d = snapped.distance(pt)
        if d.min() <= 2 * GRID_SPACING:
            key_to_id[int(key)] = int(snapped.loc[d.idxmin(), "outlet_id"])
    nodes["outlet_id"] = nodes["basin_key"].map(key_to_id)
    nodes = nodes[nodes.outlet_id.notna()].astype({"outlet_id": int})
    chan_lines = gpd.read_file(f"{RUN_NAME}_chi_data_map_lines.fgb").to_crs(EPSG)
    chan_lines["outlet_id"] = chan_lines["basin_key"].map(key_to_id)
    chan_lines = chan_lines[chan_lines.outlet_id.notna()]

basins["complete"] = basins.outlet_id.isin(nodes.outlet_id.unique() if len(nodes) else [])

summary = basins[["outlet_id", "area_km2", "complete"]].copy()
summary["snap_distance_m"] = snapped["snap_distance_m"].round(0).values
summary["n_channel_pixels"] = [int((nodes.outlet_id == i).sum()) if len(nodes) else 0
                               for i in summary.outlet_id]
print(summary.sort_values("outlet_id").to_string(index=False))

missing = sorted(set(outlets.outlet_id) - set(basins.outlet_id))
if missing:
    print(f"\nWARNING: no basin was made for outlet(s) {missing}. "
          "Scroll up to the lsdtt-basin-extraction messages to see why.")
incomplete = sorted(basins.loc[~basins.complete, "outlet_id"])
if incomplete:
    print(f"\nWARNING: basin(s) {incomplete} touch the edge of the DEM, so they are incomplete "
          "and have no channels or profile. Draw a bigger polygon (or increase BUFFER_KM) "
          "so that the whole basin fits inside, and run the notebook again from the settings.")
spill = sorted(basins.loc[~basins.within(aoi_utm.iloc[0].buffer(GRID_SPACING)), "outlet_id"])
if spill:
    print(f"\nNOTE: basin(s) {spill} extend beyond your polygon. That is fine as long as they "
          "are complete, but draw a bigger polygon next time if you want them inside it.")
far = summary[summary.snap_distance_m > 500]
if len(far):
    print(f"\nNOTE: outlet(s) {list(far.outlet_id)} were moved more than 500 m by snapping. "
          "Check on the map that the basin is the one you meant.")
if len(nodes) == 0:
    raise RuntimeError("No complete basins, so there are no channels to plot. See the warnings above.")

# Save the profile data with the outlet numbers, for use in other software
nodes.to_csv(f"{RUN_NAME}_profiles.csv", index=False)
print(f"\nChannel profile data written to {RUN_NAME}_profiles.csv")

## Colours

Each basin keeps the same colour on the map and in the profile plot. There are 8 colours, chosen so that they stay distinguishable for people with colour-blindness. With more than 8 basins, all basins share one colour and are told apart by their number instead.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
           "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
ids_with_basins = sorted(basins.outlet_id.unique())
if len(ids_with_basins) <= len(PALETTE):
    COLOURS = {oid: PALETTE[i] for i, oid in enumerate(ids_with_basins)}
else:
    COLOURS = {oid: PALETTE[0] for oid in ids_with_basins}

## Map: basins and channels over the hillshade

We use `lsdviztools3`:
* `render_hillshade` draws the hillshade in grey (`shade=False`, because it is already a hillshade).
* `render_channels` draws lines. We use it for the outline of your polygon (dashed) and for the channels (dark blue; the main stem of each basin, the longest channel from source to outlet, is thicker).
* `render_basins` draws each basin as a tinted polygon. The number next to each outlet is the number of the outlet point (1, 2, 3 ... in the order of your file).
* `render_points` marks the outlets: black crosses where you clicked, white circles where they were snapped to the channel.

Everything is drawn in the UTM coordinate system chosen above (`coordinates=CRS_STR`), so all layers line up. The axes are in kilometres.

In [ ]:
from dataclasses import replace
from IPython.display import Image, display

from lsdviztools3.render.style import MapStyle
from lsdviztools3.render.hillshade import render_hillshade
from lsdviztools3.render.basins import render_basins
from lsdviztools3.render.channels import render_channels
from lsdviztools3.render.points import render_points
from lsdviztools3.render.base import save_figure, gdf_to_display

style = MapStyle(figure_size="15c", transparent=False)

# Grey hillshade base map
fig = render_hillshade(f"{RUN_NAME}_hillshade.tif", coordinates=CRS_STR, shade=False,
                       style=replace(style, cmap="gray"))

# Your area polygon, dashed
aoi_outline = gpd.GeoDataFrame(geometry=aoi_utm.boundary, crs=EPSG)
fig = render_channels(aoi_outline, coordinates=CRS_STR, fig=fig,
                      style=replace(style, line_width="1p", line_color="black,-"))

# Basins, biggest first so that nested basins stay visible
for _, b in basins.sort_values("area_km2", ascending=False).iterrows():
    colour = COLOURS[b.outlet_id]
    fig = render_basins(basins[basins.outlet_id == b.outlet_id], coordinates=CRS_STR, fig=fig,
                        fill=colour, transparency=60,
                        style=replace(style, line_width="1p", line_color=colour))

# Channels: tributaries thin, main stems thick
main_keys = nodes.loc[nodes.groupby("outlet_id").flow_distance.idxmin(), "source_key"]
is_main = chan_lines.source_key.isin(main_keys)
for subset, width in [(chan_lines[~is_main], "0.5p"), (chan_lines[is_main], "1.5p")]:
    if len(subset):
        fig = render_channels(subset, coordinates=CRS_STR, fig=fig,
                              style=replace(style, line_width=width, line_color="#0d366b"))

# Outlets: where you clicked (crosses) and where they were snapped to (circles)
fig = render_points(outlets_utm, coordinates=CRS_STR, fig=fig, symbol="x", pen="1.2p,black",
                    style=replace(style, point_size="0.3c"))
fig = render_points(snapped, coordinates=CRS_STR, fig=fig, symbol="c", pen="0.8p,black",
                    style=replace(style, point_size="0.22c", line_color="white"))

# Basin numbers next to their outlets, drawn last so no fill covers them. The map
# axes are in km, so gdf_to_display converts the label positions from metres.
label_pts = gdf_to_display(snapped, snapped.crs)
fig.text(x=label_pts.geometry.x.values, y=label_pts.geometry.y.values,
         text=[str(i) for i in label_pts.outlet_id], justify="BL", offset="0.15c/0.15c",
         font="11p,Helvetica-Bold,black=~1.5p,white")

MAP_PNG = f"{RUN_NAME}_basins_channels_map.png"
save_figure(fig, MAP_PNG, style=style)
display(Image(MAP_PNG, width=750))

## Long profiles

A **long profile** (longitudinal profile) is the elevation of a river plotted against distance along it. Here the distance is measured upstream from each basin's outlet, so every basin starts at 0 km on the left and the channel heads are on the right.

For each basin, the thick line is the main stem and the thin lines are its tributaries, which branch off the main stem where they join it. We plot them with `matplotlib` straight from the chi data map CSV (`lsdviztools3` has plots of elevation against chi, but not against distance).

Perfectly flat stretches are usually lakes, or pits in the DEM that were filled in Step 2 so that water can flow across them.

Look for steps and changes in steepness along the profiles (*knickpoints*). They can mark changes in rock type, tectonics, or, in formerly glaciated areas like the Scottish Highlands, the lips of glacial hanging valleys.

In [ ]:
import matplotlib.pyplot as plt


def plot_basin_profile(ax, sub, colour, label=None):
    main_key = sub.loc[sub.flow_distance.idxmin(), "source_key"]
    for key, channel in sub.groupby("source_key"):
        if key == main_key:
            continue
        channel = channel.sort_values("flow_distance")
        ax.plot(channel.flow_distance / 1000, channel.elevation,
                color=colour, linewidth=0.6, alpha=0.5)
    main = sub[sub.source_key == main_key].sort_values("flow_distance")
    ax.plot(main.flow_distance / 1000, main.elevation, color=colour, linewidth=2, label=label)


def tidy(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="0.9", linewidth=0.6)
    ax.set_axisbelow(True)


profile_ids = sorted(nodes.outlet_id.unique())
if len(profile_ids) <= len(PALETTE):
    # All basins on one plot, one colour per basin
    fig_p, ax = plt.subplots(figsize=(9, 5.5))
    for oid in profile_ids:
        plot_basin_profile(ax, nodes[nodes.outlet_id == oid], COLOURS[oid], label=f"Basin {oid}")
    ax.set_xlabel("Distance upstream from basin outlet (km)")
    ax.set_ylabel("Elevation (m)")
    ax.legend(frameon=False, title="Thick: main stem\nThin: tributaries", alignment="left")
    tidy(ax)
else:
    # Many basins: one small panel per basin
    ncols = 3
    nrows = int(np.ceil(len(profile_ids) / ncols))
    fig_p, axes = plt.subplots(nrows, ncols, figsize=(11, 3 * nrows), squeeze=False)
    for ax, oid in zip(axes.flat, profile_ids):
        plot_basin_profile(ax, nodes[nodes.outlet_id == oid], COLOURS[oid])
        ax.set_title(f"Basin {oid}", loc="left")
        tidy(ax)
    for ax in axes.flat[len(profile_ids):]:
        ax.set_visible(False)
    fig_p.supxlabel("Distance upstream from basin outlet (km)")
    fig_p.supylabel("Elevation (m)")

fig_p.tight_layout()
PROFILE_PNG = f"{RUN_NAME}_long_profiles.png"
fig_p.savefig(PROFILE_PNG, dpi=200, bbox_inches="tight")
plt.show()

## Download your results (optional)

This puts every file starting with your `RUN_NAME` (the DEM, hillshade, basins, channels, profile data, the two figures and the parameter files) into one zip file and downloads it to your computer.

In [ ]:
import glob
import zipfile

ZIP_NAME = f"{RUN_NAME}_outputs.zip"
to_zip = [f for f in sorted(glob.glob(f"{RUN_NAME}_*")) if f != ZIP_NAME]
with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in to_zip:
        zf.write(name)
print(f"{ZIP_NAME}: {len(to_zip)} files, {os.path.getsize(ZIP_NAME) / 1e6:.1f} MB")

try:
    from google.colab import files
    files.download(ZIP_NAME)
except ImportError:
    print("Not running on Colab: the zip file is in", os.path.abspath(ZIP_NAME))

## Things to try

* Change `THRESHOLD_CONTRIBUTING_PIXELS` (for example to 200 or 2000) and run the notebook again from the settings cell. How do the channel networks and the number of tributaries in the profiles change?
* Add an outlet partway down one of your rivers. Its basin sits inside the bigger one (it is *nested*), and its profile is part of the bigger basin's profile.
* Compare basins of different sizes, or in different rock types. Are the long profiles smooth and concave (steep near the top, gentle near the outlet), or do they have steps?